# מפגש 23 — פתרון Audited למורה בלבד
תואם למעבדת התלמיד: flipper_length_mm, body_mass_g, island, seed17. קוד זה מציג את הבחירה רק לאחר הרצה.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score
df = pd.read_csv("https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv")
assert len(df)==344
numeric=["flipper_length_mm","body_mass_g"]
categorical=["island"]
X=df[numeric+categorical].copy()
y=df["species"].copy()
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=17,stratify=y)
assert len(X_train)==258 and len(X_test)==86

In [ ]:
numeric_pipe=Pipeline([("fill",SimpleImputer(strategy="median")),("scale",StandardScaler())])
categorical_pipe=Pipeline([("fill",SimpleImputer(strategy="most_frequent")),("encode",OneHotEncoder(handle_unknown="ignore",sparse_output=False))])
prepare=ColumnTransformer([("numeric",numeric_pipe,numeric),("categorical",categorical_pipe,categorical)])
pipe=Pipeline([("prepare",prepare),("knn",KNeighborsClassifier(weights="uniform",metric="euclidean"))])

In [ ]:
candidate_k=[3,5,9]
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
search=GridSearchCV(pipe,{"knn__n_neighbors":candidate_k},scoring="f1_macro",cv=cv,refit=True,return_train_score=True,n_jobs=1,error_score="raise")
search.fit(X_train,y_train)
table=pd.DataFrame(search.cv_results_)
assert len(table)==3
assert all(table["mean_test_score"].between(0,1))
assert search.best_params_["knn__n_neighbors"] in candidate_k
print(table[["param_knn__n_neighbors","mean_train_score","mean_test_score","std_test_score","rank_test_score"]].to_string(index=False))

In [ ]:
baseline=DummyClassifier(strategy="most_frequent")
baseline_scores=cross_val_score(baseline,X_train,y_train,cv=cv,scoring="f1_macro")
baseline.fit(X_train,y_train)
prediction=search.predict(X_test)
dummy_prediction=baseline.predict(X_test)
assert len(prediction)==len(y_test)==86
print("Best parameter:",search.best_params_)
print("CV best macro-F1:",search.best_score_)
print("CV baseline macro-F1:",baseline_scores.mean())
print("Outer Test macro-F1:",f1_score(y_test,prediction,average="macro"))
print("Outer Test baseline macro-F1:",f1_score(y_test,dummy_prediction,average="macro"))
print("Outer Test Accuracy:",accuracy_score(y_test,prediction))

In [ ]:
assert len(candidate_k)*cv.n_splits==15
assert "mean_test_score" in table.columns
assert search.best_estimator_ is not None
assert 0<=search.best_score_<=1
print("Checks passed.")